# Phase 5: Production Molecular Dynamics on Google Colab (T4 / V100 / A100 GPU)
**Project:** tRNA ASL Structural Stability Study (`trna-asl-md-stability`)  
**Objective:** Execute 6 independent 20 ns production MD replicates (3 WT + 3 CUA = 120 ns total) under NPT (310 K, 1 bar).

### Instructions:
1. In Google Colab, select **Runtime > Change runtime type > T4 GPU** (or A100/V100 if available).
2. Run the cells sequentially below.
3. At the end, download the generated trajectory and metadata archives.

## 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi

## 2. Install OpenMM & MD Dependencies

In [ ]:
!pip install -q openmm pyyaml mdtraj numpy

## 3. Clone / Sync the Repository

In [ ]:
import os
if not os.path.exists('trna-asl-md-stability'):
    !git clone https://github.com/nitinjadhav888/trna-asl-md-stability.git
%cd trna-asl-md-stability
!git pull origin main

## 4. Run Isolated Pipeline Validation Test
Verifies OpenMM platform, DCD writing, checkpoint restart, and simulated-time tracking before running production.

In [ ]:
!python scripts/07_run_production.py --test-pipeline

## 5. Execute Production Replicates (20 ns each)
On a Colab T4 GPU, each 20 ns replicate takes ~1.5 hours (~300 ns/day).  
You can run all 6 sequentially, or run individual replicates across multiple Colab tabs.

In [ ]:
# Run Wild-Type Replicates (Seeds: 5101, 5102, 5103)
!python scripts/07_run_production.py --construct wt --replicate 1 --duration-ns 20.0
!python scripts/07_run_production.py --construct wt --replicate 2 --duration-ns 20.0
!python scripts/07_run_production.py --construct wt --replicate 3 --duration-ns 20.0

In [ ]:
# Run CUA Suppressor Replicates (Seeds: 5201, 5202, 5203)
!python scripts/07_run_production.py --construct cua --replicate 1 --duration-ns 20.0
!python scripts/07_run_production.py --construct cua --replicate 2 --duration-ns 20.0
!python scripts/07_run_production.py --construct cua --replicate 3 --duration-ns 20.0

## 6. Run-Level Automated Quality Control Verification
Verifies atom counts, sequence integrity, continuous time (>= 20 ns), temperature, density, and trajectory readability.

In [ ]:
!python scripts/07_run_production.py --validate-qc --duration-ns 20.0

## 7. Package Outputs for Local Download
Creates a compressed tarball containing all production logs, metadata, thermodynamic CSVs, checkpoints, and DCD trajectories.

In [ ]:
!tar -czvf phase5_production_outputs.tar.gz simulations/*_rep*
from google.colab import files
files.download('phase5_production_outputs.tar.gz')